In [7]:
import torch
import numpy as np
import nnsight
from pathlib import Path
from int_utils import ATTENTION_INTERVENTIONS_DIR, PatchTarget, Config
from int_utils import run_experiment, EXAMPLES_PATH, test_fn

In [8]:
def patch_heads(cfg: Config,
                base: str,
                source: str,
                source_activations):

    with torch.no_grad(), cfg.model.trace() as tracer:
        barriers = [tracer.barrier(cfg.n_heads + 1)
                    for _ in range(cfg.n_layers)]

        with tracer.invoke(source):
            for layer_idx in range(cfg.n_layers):
                src_w = (cfg.model.transformer.h[layer_idx].attn.source
                            .attention_interface_1.output[1].clone())

                barriers[layer_idx]()

        module_effects = []
        for layer_idx in range(cfg.n_layers):
            layer_effects = []

            for head_idx in range(cfg.n_heads):
                with tracer.invoke(base):

                    src = cfg.model.transformer.h[layer_idx].attn.source
                    ai = src.attention_interface_1
                    V = src.value_states_2.output

                    barriers[layer_idx]()

                    w = ai.output[1].clone()
                    w[:, head_idx] = src_w[:, head_idx]

                    new_out = torch.matmul(w, V).transpose(1, 2)
                    ai.output = (new_out, w)

                    logits = cfg.model.output.logits[:, -1, :]
                    m = (logits[:, cfg.she_tk] -
                         logits[:, cfg.he_tk]).save()

                    layer_effects.append(m.detach().cpu())
            module_effects.append(layer_effects)

        module_effects = nnsight.save(module_effects)

    arr = np.array(module_effects).squeeze()
    return arr


def patch_layers(cfg: Config,
                 base: str,
                 source: str,
                 source_activations):

    with torch.no_grad(), cfg.model.trace() as tracer:
        barriers = [tracer.barrier(2)
                    for _ in range(cfg.n_layers)]

        with tracer.invoke(source):
            for layer_idx in range(cfg.n_layers):
                src_w = (cfg.model.transformer.h[layer_idx].attn.source
                            .attention_interface_1.output[1].clone())

                barriers[layer_idx]()

        module_effects = []
        for layer_idx in range(cfg.n_layers):
            layer_effects = []

            with tracer.invoke(base):
                src = cfg.model.transformer.h[layer_idx].attn.source
                ai = src.attention_interface_1
                V = src.value_states_2.output

                barriers[layer_idx]()

                w = ai.output[1].clone()
                w[:] = src_w[:]

                new_out = torch.matmul(w, V).transpose(1, 2)
                ai.output = (new_out, w)

                logits = cfg.model.output.logits[:, -1, :]
                m = (logits[:, cfg.she_tk] -
                     logits[:, cfg.he_tk]).save()

                layer_effects.append(m.detach().cpu())

            module_effects.append(layer_effects)

        module_effects = nnsight.save(module_effects)

    arr = np.array(module_effects).squeeze()
    return arr


def patch_model(cfg:Config,
                base: str,
                source: str,
                source_activations):
    with torch.no_grad(), cfg.model.trace(source):
        src_ws = nnsight.save([
            cfg.model.transformer.h[l].attn.source
            .attention_interface_1.output[1].clone()
            for l in range(cfg.n_layers)
        ])

    with torch.no_grad(), cfg.model.trace(base):
        for l in range(cfg.n_layers):
            src = cfg.model.transformer.h[l].attn.source
            ai = src.attention_interface_1
            V = src.value_states_2.output

            w = src_ws[l].clone()
            ai.output = (torch.matmul(w, V).transpose(1, 2), w)

        logits = cfg.model.output.logits[:, -1, :]
        m = (logits[:, cfg.she_tk] - logits[:, cfg.he_tk]).save()

    return m.detach().cpu().squeeze()

In [ ]:
# test_fn(patch_model)

In [10]:
BASE_DIR: Path = ATTENTION_INTERVENTIONS_DIR
for d in ["heads", "layers", "model"]:
    (BASE_DIR / d).mkdir(parents=True, exist_ok=True)


PATCH_TARGETS = {
    "heads": PatchTarget(patch_fn=patch_heads,
                         output_dir=BASE_DIR / "heads",
                         effect_shape=(12, 12)),
    "layers": PatchTarget(patch_fn=patch_layers,
                          output_dir=BASE_DIR / "layers",
                          effect_shape=(12,)),
    "model": PatchTarget(patch_fn=patch_model,
                         output_dir=BASE_DIR / "model",
                         effect_shape=()),
}


for n, patch_target in PATCH_TARGETS.items():
    print(f"{'*' * 50}\nPatching {n}\n{'*' * 50}")
    run_experiment(EXAMPLES_PATH, patch_target)

**************************************************
Patching heads
**************************************************


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Device: cpu
Loaded 2873 processed IDs.
man_direct: 2873 already saved; 0 remaining
Loaded 2873 processed IDs.
man_indirect: 2873 already saved; 0 remaining
Loaded 2873 processed IDs.
woman_direct: 2873 already saved; 0 remaining
Loaded 2873 processed IDs.
woman_indirect: 2873 already saved; 0 remaining
**************************************************
Patching layers
**************************************************


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Device: cpu
man_direct: 0 already saved; 2873 remaining
man_direct: saved examples 5/2873
man_direct: saved examples 10/2873
man_direct: saved examples 15/2873
man_direct: saved examples 20/2873


KeyboardInterrupt: 